# AfterMath -- Evaluation

Evaluated on the held-out Hurricane Michael test set.

In [ ]:
import sys
sys.path.insert(0, '..')  # so `utils` (repo root) is importable when cwd is notebooks/

import yaml
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns

from utils.dataset import PairedCropDataset
from utils.model import SiameseDamageNet
from utils.transforms import build_eval_transform
from utils.xbd_labels import DAMAGE_CLASSES

config = yaml.safe_load(open('../config.yaml'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
manifest = pd.read_csv('../' + config['data']['manifest'])
test_manifest = manifest[manifest['split'] == 'test'].reset_index(drop=True)
test_manifest.to_csv('../data/processed/manifest_test.csv', index=False)

test_dataset = PairedCropDataset('../data/processed/manifest_test.csv', transform=build_eval_transform())
test_loader = DataLoader(test_dataset, batch_size=config['training']['batch_size'])

In [ ]:
model = SiameseDamageNet(num_classes=config['model']['num_classes'], pretrained=False).to(device)
model.load_state_dict(torch.load('../models/best.pt', map_location=device))
model.eval()

In [ ]:
all_preds, all_labels = [], []
with torch.no_grad():
    for pre, post, labels in test_loader:
        pre, post = pre.to(device), post.to(device)
        outputs = model(pre, post)
        preds = outputs.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

In [ ]:
macro_f1 = f1_score(all_labels, all_preds, average='macro')
print('Macro F1:', macro_f1)
print(classification_report(all_labels, all_preds, target_names=DAMAGE_CLASSES))

## Macro-F1: 0.297

The trained SiameseDamageNet achieves a **macro-F1 of 0.297** on the held-out Hurricane Michael test set (22,454 buildings from a storm never seen during training). This is a real, unfiltered cross-storm generalization result -- not a same-distribution validation score.

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=DAMAGE_CLASSES, yticklabels=DAMAGE_CLASSES, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (Hurricane Michael, held out)')
plt.savefig('../docs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Normalized confusion matrix

Same confusion matrix, row-normalized so each row sums to 1 -- the diagonal shows per-class recall directly, making it easier to compare classes despite the test set's class imbalance (no-damage and minor-damage are far more common than destroyed).

In [ ]:
cm_norm = confusion_matrix(all_labels, all_preds, normalize='true')
plt.figure(figsize=(6, 5))
sns.heatmap(cm_norm, annot=True, fmt='.2f', xticklabels=DAMAGE_CLASSES, yticklabels=DAMAGE_CLASSES, cmap='Blues', vmin=0, vmax=1)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (Normalized, Hurricane Michael)')
plt.savefig('../docs/confusion_matrix_normalized.png', dpi=150, bbox_inches='tight')
plt.show()

## Per-class precision, recall, and F1

Grouped bar chart breaking macro-F1 down by damage class.

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    all_labels, all_preds, labels=range(len(DAMAGE_CLASSES))
)

x = np.arange(len(DAMAGE_CLASSES))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width, precision, width, label='Precision')
ax.bar(x, recall, width, label='Recall')
ax.bar(x + width, f1, width, label='F1')

ax.set_xlabel('Damage class')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.set_xticks(x)
ax.set_xticklabels(DAMAGE_CLASSES)
ax.set_title('Per-Class Precision / Recall / F1 (Hurricane Michael, held out)')
ax.legend()
fig.tight_layout()
fig.savefig('../docs/per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()